<a href="https://colab.research.google.com/github/GayoungJeong/customer-segmentation/blob/main/notebooks/02_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/marketing-portfolio/data/retail.csv')

clean = df.copy()    # 원본 df는 두고 복사본에서 작업
print(len(clean))    # 1067371

Mounted at /content/drive
1067371


In [2]:
clean = clean[~clean['Description'].isnull()]   # 빈칸 찾는 함수 (11/14 메모 참고)
print(len(clean))

1062989


In [3]:
clean = clean[~clean['Invoice'].astype(str).str.startswith('A')]
print(len(clean))

1062983


가격이 음수인 것들은 모두 Invoice A로 시작(5개), 그런데 Invoice A로 시작하는 열이 6개임

In [17]:
df[df['Invoice'].astype(str).str.startswith('A')]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
825443,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


확인해 보니 대손 처리 기록인 것으로 추정 (Invoice가 A로 시작, 6줄. 모두 “Adjust bad debt”이며 같은 금액의 +/- 정정 기록 포함)
어쨌든 실제 상품구매 기록이 아니므로 분석에서 제외

| 단계 | 지운 이유 | 남은 행 수 |
|---|---|---|
| 원본 | - | 1,067,371 |
| 1단계 | 상품명 없음 (재고 조정 기록으로 추정) | 1,062,989 |
| 2단계 | 대손 처리 기록 (Invoice가 A로 시작) | 1,062,983 |

In [4]:
clean = clean[~clean['Customer ID'].isnull()] #고객아이디 없는 줄 지우기
print(len(clean)) #여기부터 고객 분석용 데이터

824364


In [9]:
clean['TotalPrice'] = clean['Quantity'] * clean['Price']
clean.head()   # 맨 오른쪽에 TotalPrice 열이 생겼는지 확인

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [10]:
clean[clean['Customer ID'] == 16446]['TotalPrice'].sum()

np.float64(-6.100000000005821)

In [15]:
clean[clean['Customer ID'] == 16446]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
371780,C525275,TEST001,This is a test product.,-2,2010-10-04 16:38:00,4.50,16446.0,United Kingdom,-9.00
719815,553573,22980,PANTRY SCRUBBING BRUSH,1,2011-05-18 09:52:00,1.65,16446.0,United Kingdom,1.65
719816,553573,22982,PANTRY PASTRY BRUSH,1,2011-05-18 09:52:00,1.25,16446.0,United Kingdom,1.25
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,168469.60
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom,-168469.60


취소건만 지우면 그것과 짝이 되는 품목을 같이 지울 수 없다는 문제가 생기는데, 그냥 customer ID기준으로 구매한 (가격*수량)을 더하면 한 사람이 얼마를 소비했는지 알 수 있게 됨

In [18]:
clean.to_csv('/content/drive/MyDrive/marketing-portfolio/data/retail_clean.csv', index=False)

| 단계 | 지운 이유 | 남은 행 수 |
|---|---|---|
| 원본 | - | 1,067,371 |
| 1단계 | 상품명 없음 (재고 조정 기록으로 추정) | 1,062,989 |
| 2단계 | 대손 처리 기록 (Invoice가 A로 시작) | 1,062,983 |
| 3단계 | 고객 ID 없음 (누가 샀는지 몰라 고객 분석 불가) | 824,364 |

※ 취소 건은 삭제하지 않음. 수량이 음수라 금액도 음수가 되므로, 고객별로 금액을 합산하면 주문과 취소가 자동으로 상쇄됨. (예: 16446번 고객은 80,995개 주문 후 취소 → 합계 약 0)